**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**27 | Final HARTH Research Evidence Synthesis and Reproducibility Report**

![Final research evidence synthesis workflow](https://raw.githubusercontent.com/RezaSaadatyar/human-movement-intelligence/main/wearable_activity_recognition/images/27_workflow.svg)

**Objective:** Consolidate recorded evidence from HARTH Stages 23–26 and examine its provenance, evaluation boundaries, model comparisons and reproducibility.

**Evidence Sources:** Independently saved, versioned CSV results in `wearable_activity_recognition/output/` and the approved and reconciled protocol JSON manifests. The displayed-precision Stage 23/24/26 CSV snapshots were transcribed once from previously completed research outputs; no earlier notebook is parsed or executed for this report. This Notebook does not download raw HARTH recordings, load model weights or access reserved Test source files.

**Notebook Scope:** Read named, standalone output files for the Development benchmark, robustness checks and previously approved qualified held-out results; use the frozen JSON manifests to describe model selection. No retraining, rescoring or reading previous notebook structure occurs. Stages 23–24 report Development results; Stage 26 reports qualified same-study held-out results for two participants previously inspected descriptively, not blind external validation.

**Evidence checks:** Confirm required input files exist, retain their SHA256 digests in report provenance, check the unchanged approved protocol and basic report-table coverage. Use only recorded displayed precision (usually four decimal places); do not reconstruct or invent missing metrics.

**00. Colab Repository Setup**

Prepare the repository using the Code Cell below. The report reads recorded results without training or evaluating the models again.

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content
/content/human-movement-intelligence


**01. Imports and Evidence-Reading Utilities**

In [ ]:
# ================ 01. Imports and Evidence-Reading Utilities =================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ---------------------------- 2. Standard Library ----------------------------
import hashlib
import json
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4))

# ------------------------ 5. Study Output and Protocol Paths -----------------
base = root / "wearable_activity_recognition"
output_dir = base / "output"
output_dir.mkdir(parents=True, exist_ok=True)
manifest_pending = base / "evidence/stage_25_protocol_proposal_reconciled.json"
manifest_approved = base / "evidence/stage_25_protocol_approved_qualified_test.json"

# --------------------- 6. Read Named Reusable Result Tables ------------------
# Displayed-precision CSV snapshots are committed research reports, NOT
# reconstructed full-precision predictions or a new Test evaluation.
input_paths = {
    "benchmark": output_dir / "23_benchmark_model_summary_display.csv",
    "robustness_summary": output_dir / "24_robustness_model_summary_display.csv",
    "robustness_folds": output_dir / "24_robustness_fold_metrics_display.csv",
    "qualified_coverage": output_dir / "26_qualified_subject_coverage_display.csv",
    "qualified_summary": output_dir / "26_qualified_model_summary_display.csv",
    "qualified_subjects": output_dir / "26_qualified_per_subject_display.csv",
    "qualified_classes": output_dir / "26_qualified_per_class_display.csv",
}
for name, path in input_paths.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing reusable evidence file: {path}. "
            "Restore the committed output CSV; do not rerun the approved Test."
        )
tables = {name: pd.read_csv(path) for name, path in input_paths.items()}
benchmark = tables["benchmark"]
robustness_summary = tables["robustness_summary"]
robustness_folds = tables["robustness_folds"]
qualified_coverage = tables["qualified_coverage"]
qualified_summary = tables["qualified_summary"]
qualified_subjects = tables["qualified_subjects"]
qualified_classes = tables["qualified_classes"]

# ---------------------- 7. Read Approved Frozen Protocol ---------------------
pending_protocol = json.loads(manifest_pending.read_text(encoding="utf-8"))
approved_protocol = json.loads(manifest_approved.read_text(encoding="utf-8"))
assert pending_protocol["status"] == "PENDING_USER_REVIEW"
assert approved_protocol["status"] == "APPROVED"
assert approved_protocol.get("approval_record")
assert {k: v for k, v in pending_protocol.items() if k != "status"} == {
    k: v for k, v in approved_protocol.items()
    if k not in ("status", "approval_record")
}

# ---------------------- 8. Validate Available Report Tables ------------------
assert set(benchmark["model"]) == {
    "Random Forest", "Standard CNN", "Weighted CNN", "TCN",
    "Multi-Scale CNN", "1D-ResNet", "Time-Series Transformer",
}
assert len(robustness_folds) == 36 and len(robustness_summary) == 4
assert set(robustness_folds["seed"]) == {42, 123, 2026}
assert set(robustness_folds["fold"]) == {1, 2, 3}
assert qualified_coverage["sampled_windows"].tolist() == [500, 500]
assert qualified_coverage["subject_id"].tolist() == ["S023", "S032"]
assert qualified_summary["model"].tolist() == [
    "1D-ResNet Weighted CE", "Random Forest"
]
assert len(qualified_subjects) == 4 and len(qualified_classes) == 16
assert set(qualified_classes["source_code"]) == set(range(1, 9))
assert qualified_classes.groupby("model")["support"].sum().eq(1000).all()

print("Reusable output CSV files and unchanged approved protocol: VERIFIED")
print("Notebook structures or saved HTML inspected: NO")
print("Raw HARTH data, retraining or Test re-scoring: NONE")
print("Research output directory:", output_dir)

**02. Dataset, Cohort and Experimental Design**

The source is pinned HARTH v2.0 from the NTNU AI Lab source repository, not the older UCI HARTH release. Six lower-back/right-thigh accelerometer channels sampled at nominal 50 Hz are windowed to 100 samples with stride 50, strict annotation purity and continuity. Eight original documented class codes are used; undocumented codes are not given invented activity names. A label-independent cap of 500 windows per person yields 14,201 Development windows from 29 people. Test uses 500 eligible windows each from S023 and S032, who had been inspected in earlier exploration. Correlated overlapping windows are not independent observations.

In [ ]:
# ================ 02. Dataset, Cohort and Experimental Design ================

source = approved_protocol["source"]
primary = approved_protocol["primary_model"]
reference = approved_protocol["reference_model"]
protocol_rows = [
    ("Source dataset", source["dataset"]),
    ("Pinned revision", source["revision"]),
    ("Development people", source["development_participants"]),
    ("Development windows", source["expected_development_windows"]),
    ("Reserved Test people", ", ".join(source["reserved_test_people"])),
    ("Test cap per person", source["test_cap_per_person"]),
    ("Sampling frequency (Hz)", source["sampling_rate_hz"]),
    ("Window / stride (samples)", f'{source["window_size"]} / {source["step_size"]}'),
    ("Classification codes", str(source["task_codes"])),
    ("Primary network", primary["architecture"]),
    ("Fixed optimiser updates", primary["optimizer_steps"]),
    ("Class weighting", primary["training_loss"]),
    ("Random Forest trees", reference["n_estimators"]),
]
display(pd.DataFrame(protocol_rows, columns=["Protocol component", "Recorded setting"]))
display(qualified_coverage)
assert qualified_coverage["sampled_windows"].sum() == 1000


**03. Seven-Model Matched Development Comparison (Stage 23)**

Seven predeclared classifiers were evaluated using the same 29 Development participants and three subject-disjoint folds. Compare saved mean Macro F1, observed variability, accuracy and computational cost. Architecture, features and training-loss choice can differ between systems; a ranking is descriptive, not a causal architecture-only experiment.

In [ ]:
# ========= 03. Seven-Model Matched Development Comparison (Stage 23) =========

# -------------------- 1. Seven-Model Development Results ---------------------
benchmark = benchmark.sort_values("macro_f1_mean", ascending=False)
assert len(benchmark) == 7
display(benchmark.round(4))

# --------------------- 2. Fixed-Task Macro F1 Comparison ---------------------
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(benchmark["model"][::-1], benchmark["macro_f1_mean"][::-1],
        xerr = benchmark["macro_f1_sd"][::-1], capsize=3)
ax.set_xlabel("Development participant-held-out Macro F1 (mean ± fold SD)")
ax.set_title("Stage 23 | Seven-model matched Development benchmark")
ax.set_xlim(0, 1)
fig.tight_layout()
save_figure(fig, output_dir / "27_development_seven_model_comparison.png")
plt.show()


**04. Repeated-Seed Robustness and Training-Loss Comparisons (Stage 24)**

Stage 24 evaluated 36 model/fold/seed jobs using seeds 42, 123 and 2026 across three disjoint Development folds. Report all four selected model/loss conditions rather than only the winning configuration. Across-seed SD refers to variation among three seed-level fold averages; it is not a confidence interval or independent cohort replication.

In [ ]:
# === 04. Repeated-Seed Robustness and Training-Loss Comparisons (Stage 24) ===

# --------------------- 1. Multi-Seed and Loss Robustness ---------------------
robust = robustness_summary.sort_values(
    "mean_across_seed_fold_macro_f1", ascending=False
)
assert len(robustness_folds) == 36 and len(robust) == 4
display(robust.round(4))

# ---------------------- 2. Seed Stability Visualisation ----------------------
labels = robust["architecture"].astype(str)+" | "+robust["loss_method"].astype(str)
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(labels[::-1], robust["mean_across_seed_fold_macro_f1"][::-1],
        xerr = robust["across_seed_sd"][::-1], capsize=3)
ax.set_xlim(0, 1)
ax.set_xlabel("Development Macro F1 (seed/fold mean ± across-seed SD)")
ax.set_title("Stage 24 | Repeated-seed robustness and loss comparison")
fig.tight_layout()
save_figure(fig, output_dir / "27_development_multiseed_robustness.png")
plt.show()


**05. Frozen Model Selection and Development Fit**

The selected primary was a 1D-ResNet with inverse-square-root weighted cross-entropy, 400 Adam updates, batch size 128, learning rate 0.001 and seed 42. The reference was a 120-tree Random Forest with 30 statistical features and fixed hyperparameters. Final fitting used 29 Development people only, without Test tuning. The saved Stage 26 Development inference reload check passed. Detailed model weights remain ignored local artefacts, not necessarily present in a new runtime. Stage 25 was selection on Development, not independent confirmation.

In [ ]:
# ============== 05. Frozen Model Selection and Development Fit ===============

# ------------------- 1. Frozen Development Model Selection -------------------
decision = pd.DataFrame([
    {"Role":"Preselected primary", "Model":"1D-ResNet Weighted CE",
     "Selection basis":"Development Stage 23/24"},
    {"Role":"Predeclared reference", "Model":"Random Forest",
     "Selection basis":"Development Stage 23/24"},
])
display(decision)

# ---------------------- 2. No Test-Driven Model Tuning -----------------------
assert primary["architecture"] == "1D-ResNet" and primary["optimizer_steps"] == 400
assert reference["n_estimators"] == 120
assert approved_protocol["evaluation"]["use_test_for_tuning"] is False
print("Model choice frozen before qualified Test: YES")
print("Test-driven hyperparameter/model changes permitted: NO")


**06. Qualified Held-Out Evaluation (Stage 26)**

The original reserved Test participants were S023 and S032, with 500 windows selected from eligible recordings for each person under the approved protocol. The saved Stage 26 results report fixed eight-class Macro F1 of **0.7182** for weighted 1D-ResNet and **0.6817** for Random Forest. These figures are read from the earlier evaluation, not recomputed here. Balanced accuracy averages recall over classes observed in each evaluated subset; its class denominator can differ from the fixed eight-class Macro F1 denominator. The two models show different accuracy and class-specific error profiles.

In [ ]:
# =============== 06. Qualified Held-Out Evaluation (Stage 26) ================

# ----------------- 1. Qualified Test Summary and Individuals -----------------
display(qualified_summary.round(4))
display(qualified_subjects.round(4))

# --------------------- 2. Fixed-Task Participant Scores ----------------------
fig, ax = plt.subplots(figsize=(8, 4))
pivot = qualified_subjects.pivot(index="subject_id", columns="model",
                                values = "fixed_eight_class_macro_f1")
pivot.plot.bar(ax=ax, ylim=(0, 1), rot=0)
ax.set_ylabel("Fixed eight-class Macro F1")
ax.set_xlabel("Historically inspected original Test participant")
ax.set_title("Stage 26 | Qualified participant-level descriptive evaluation")
ax.legend(title="Frozen model")
fig.tight_layout()
save_figure(fig, output_dir / "27_qualified_participant_macro_f1.png")
plt.show()


**07. Activity-Level Errors, Actual Support and Class Imbalance**

Report precision, recall, F1 and true support for all eight source classes. The qualified held-out subset contains only **3 Shuffling**, **10 Stairs ascending** and **6 Stairs descending** windows. High recall on such small groups is unstable evidence and does not establish reliable recognition for unseen participants. Confusion matrices remain in Notebook 26; their counts are not inferred from rounded metrics.

In [ ]:
# ======= 07. Activity-Level Errors, Actual Support and Class Imbalance =======

# --------------------- 1. Eight-Class Prediction Errors ----------------------
per_class = qualified_classes.sort_values(["model", "source_code"])
display(per_class.round(4))

# ------------------------- 2. Observed Class Support -------------------------
support = per_class[per_class.model == "1D-ResNet Weighted CE"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(support["activity"], support["support"])
ax.set_ylabel("Original Test window count")
ax.set_title("Stage 26 | True activity support (1,000 selected windows)")
ax.tick_params(axis="x", rotation=40)
fig.tight_layout()
save_figure(fig, output_dir / "27_qualified_activity_support.png")
plt.show()
assert int(support["support"].sum()) == 1000
print("Rare-class evidence must be qualified; no support-based significance claim.")


**08. Source Quality, Evaluation Limitations and Claims Boundary**

The source reports undocumented HARTH codes 10 and 11 in S032. They were left uninterpreted and excluded after annotation-safe windowing. Participants S023 and S032 were descriptively inspected during earlier development exploration; they are not a historically blind or externally independent cohort. Only two people were evaluated, with overlapping source windows and participant-dependent class support. This study does not establish clinical efficacy, robustness to unseen data sources, medical deployment, population-wide performance, statistically significant superiority or causal benefit of the architecture. Development fold metrics and final qualified-Test metrics must remain separate.

In [ ]:
# ====== 08. Source Quality, Evaluation Limitations and Claims Boundary =======

boundaries = pd.DataFrame([
    ("Supported", "29-person Development, 3 subject-disjoint folds and Stage 24 3-seed robustness"),
    ("Supported", "Frozen models assessed on identical windows from S023 and S032"),
    ("Supported", "Stage 26 source-aligned class metrics and qualified same-study results"),
    ("Limitation", "Only 2 originally reserved people; both historically explored"),
    ("Limitation", "Overlapping windows, small rare-class support, different per-person class coverage"),
    ("Limitation", "Previously undocumented codes 10/11 were excluded, not reinterpreted"),
    ("Not supported", "Independent blind external validation or prospective clinical deployment"),
    ("Not supported", "Test-selected architectures, clinical decisions, population-level certainty"),
], columns=["Evidence status", "Interpretation"])
display(boundaries)


**09. Reproducible Report Export and Provenance**

Export model-level, participant-level and class-level summary tables to ignored local outputs, together with notebook hashes, the Development cohort fingerprint and the approved manifest fingerprint. No raw acceleration rows or per-window predictions are exported. The version-controlled notebooks retain the saved tables used for this report; the detailed Stage 26 prediction files remain local.

In [ ]:
# =============== 09. Reproducible Report Export and Provenance ===============

# ----------------------- 1. Save Research Synthesis Tables -------------------
pd.DataFrame(protocol_rows, columns=[
    "protocol_component", "recorded_setting",
]).to_csv(output_dir / "27_frozen_protocol_summary.csv", index=False)
decision.to_csv(output_dir / "27_frozen_model_selection.csv", index=False)
boundaries.to_csv(output_dir / "27_research_evidence_boundaries.csv", index=False)

# ------------------- 2. Report Input-File Lineage and Digests -----------------
# Input tables remain in their original output CSVs; no duplicate numerical
# results are manufactured or recovered from previous Jupyter notebooks.
lineage = {
    "study": "HARTH v2.0",
    "scope": "file-based display-precision synthesis; no retraining or Test re-evaluation",
    "source_revision": source["revision"],
    "development_sha256": approved_protocol["source_development_sha256"],
    "input_csv_sha256": {
        name: hashlib.sha256(path.read_bytes()).hexdigest()
        for name, path in input_paths.items()
    },
    "pending_manifest_sha256": hashlib.sha256(
        manifest_pending.read_bytes()
    ).hexdigest(),
    "approved_manifest_sha256": hashlib.sha256(
        manifest_approved.read_bytes()
    ).hexdigest(),
    "qualified_test_subjects": ["S023", "S032"],
    "test_already_historically_inspected": True,
    "metric_precision": "source-saved displayed table precision; no full-precision recovery",
    "notebook_parsing_or_execution": False,
}
(output_dir / "27_report_provenance.json").write_text(
    json.dumps(lineage, indent=2) + "\n", encoding="utf-8"
)
print("Report-only files written to:", output_dir)
print("Raw HARTH data opened or downloaded during Stage 27: NO")


**10. Final Evidence Quality Gate and Reporting Boundaries**

The final checks require consistent saved metrics, source provenance, approved evaluation boundaries and exported report files. A PASS marker is valid only after successful execution and saving of this report; it verifies evidence synthesis, not independent experiment replication. These checks do **not** establish peer review, external validation, statistical superiority or independent replication.

In [ ]:
# ========= 10. Final Evidence Quality Gate and Reporting Boundaries =========

# ----------------- 1. Verify File-Based Research Evidence --------------------
assert len(benchmark) == 7 and len(robust) == 4
assert len(qualified_summary) == 2 and len(qualified_subjects) == 4
assert len(qualified_classes) == 16
assert set(qualified_classes.source_code) == set(range(1, 9))
assert qualified_coverage["sampled_windows"].tolist() == [500, 500]
assert approved_protocol["status"] == "APPROVED"
assert all(path.is_file() for path in input_paths.values())
assert all((output_dir / filename).is_file() for filename in (
    "27_frozen_protocol_summary.csv",
    "27_frozen_model_selection.csv",
    "27_research_evidence_boundaries.csv",
    "27_report_provenance.json",
    "27_development_seven_model_comparison.png",
    "27_development_multiseed_robustness.png",
    "27_qualified_participant_macro_f1.png",
    "27_qualified_activity_support.png",
))

# ------------------------ 2. Report Scientific Limits -------------------------
print("=" * 76)
print("STAGE 27 | FILE-BASED RESEARCH SYNTHESIS QUALITY GATE: PASSED")
print("=" * 76)
print("Stage 23–24 Development CSV evidence: INCLUDED")
print("Stage 26 previously approved qualified evaluation CSVs: INCLUDED")
print("Seven Development comparison models: INCLUDED")
print("Thirty-six Stage 24 robustness jobs: SUMMARISED")
print("Two frozen models and both historically inspected Test people: INCLUDED")
print("Rare-class support and limitations: EXPLICIT")
print("Previous notebook HTML/cell structure inspected: NO")
print("Raw HARTH data / model fitting / Test re-scoring: NO")
print("Outputs stored:", output_dir)
